In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.base import clone 

In [2]:
df_train = pd.read_csv('../datasets/processed/train_combine_lag.csv')
ensemble_df = pd.read_csv('../result/cumulative_contribution.csv')

df_train['timestamp'] = pd.to_datetime(df_train['timestamp'])

target_kolom = 'output_hybrid'
fitur_utama = ['temperature', 'humidity', 'surface_radiation', 'upper_atmospheric_radiation', 'air_density', 'wind_velocity', 'hour', 'month']

def calculate_maape(y_true, y_pred):
    epsilon = 1e-10
    return np.mean(np.arctan(np.abs((y_true - y_pred) / (np.abs(y_true) + epsilon)))) * 100

konfigurasi_model = {
    'RidgeRegression': {
        'model': Ridge(alpha=0.1, solver='lsqr'),
        'threshold': 1.00 
    },
    'RandomForest': {
        'model': RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1),
        'threshold': 0.85  
    },
    'XGBoost': {
        'model': XGBRegressor(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=42, n_jobs=-1),
        'threshold': 0.95  
    },
    'LightGBM': {
        'model': LGBMRegressor(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1, verbose=-1),
        'threshold': 0.85  
    },
    'SVR': {
        'model': SVR(kernel='rbf', C=10, epsilon=0.01, gamma=0.01),
        'threshold': 0.95  
    }
}

tscv = TimeSeriesSplit(n_splits=5)

all_model_reports = {}

for nama_model, config in konfigurasi_model.items():
    th = config['threshold']
    if th == 1.00:
        lag_lolos = ensemble_df['Feature'].tolist()
    else:
        idx_cutoff = (ensemble_df['Cumulative_Contribution'] >= th).idxmax()
        lag_lolos = ensemble_df['Feature'].iloc[:idx_cutoff + 1].tolist()
        
    fitur_aktif_total = fitur_utama + lag_lolos
    
    X_train = df_train[fitur_aktif_total].values
    y_train = df_train[target_kolom].values
    
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    
    X_train_scaled = scaler_X.fit_transform(X_train)
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    
    cv_records = []
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_scaled), 1):
        model_cv = clone(config['model'])
        
        model_cv.fit(X_train_scaled[train_idx], y_train_scaled[train_idx])
        
        preds_val_scaled = model_cv.predict(X_train_scaled[val_idx])
        preds_val = scaler_y.inverse_transform(preds_val_scaled.reshape(-1, 1)).flatten()
        y_val_true = y_train[val_idx]
        
        fold_rmse = np.sqrt(mean_squared_error(y_val_true, preds_val))
        fold_mae = mean_absolute_error(y_val_true, preds_val)
        fold_maape = calculate_maape(y_val_true, preds_val)
        fold_r2 = r2_score(y_val_true, preds_val)
        
        cv_records.append({
            'Fold': f"Fold {fold}",
            'RMSE': fold_rmse, 
            'MAE': fold_mae, 
            'MAAPE (%)': fold_maape, 
            'R2': fold_r2
        })
        
    df_cv_summary = pd.DataFrame(cv_records)
    
    mean_values = df_cv_summary[['RMSE', 'MAE', 'MAAPE (%)', 'R2']].mean()
    std_values = df_cv_summary[['RMSE', 'MAE', 'MAAPE (%)', 'R2']].std()
    
    row_mean = pd.DataFrame([{'Fold': 'Mean', 'RMSE': mean_values['RMSE'], 'MAE': mean_values['MAE'], 'MAAPE (%)': mean_values['MAAPE (%)'], 'R2': mean_values['R2']}])
    row_std = pd.DataFrame([{'Fold': 'Std', 'RMSE': std_values['RMSE'], 'MAE': std_values['MAE'], 'MAAPE (%)': std_values['MAAPE (%)'], 'R2': std_values['R2']}])
    
    df_final_report = pd.concat([df_cv_summary, row_mean, row_std], ignore_index=True)
    
    print(f"\nCross Validation Summary Table ({nama_model})\n")
    print(df_final_report.to_string(index=False, formatters={
        'RMSE': '{:,.4f}'.format, 
        'MAE': '{:,.4f}'.format,
        'MAAPE (%)': '{:,.2f}%'.format,
        'R2': '{:,.4f}'.format
    }))
    
    all_model_reports[nama_model] = df_final_report


Cross Validation Summary Table (RidgeRegression)

  Fold     RMSE      MAE MAAPE (%)     R2
Fold 1 344.9275 240.1835    21.57% 0.9691
Fold 2 339.2626 248.1258    12.10% 0.9747
Fold 3 236.5914 175.4384    18.14% 0.9717
Fold 4 232.6059 155.9013    17.73% 0.9754
Fold 5 301.4443 201.7013    13.99% 0.9743
  Mean 290.9663 204.2701    16.70% 0.9730
   Std  54.1219  39.9703     3.72% 0.0026

Cross Validation Summary Table (RandomForest)

  Fold     RMSE      MAE MAAPE (%)     R2
Fold 1 436.6357 249.9029    13.18% 0.9505
Fold 2 412.8054 255.6310     9.47% 0.9625
Fold 3 279.2652 173.2692    13.33% 0.9605
Fold 4 250.6544 138.0112     9.61% 0.9714
Fold 5 285.2770 154.0785     6.55% 0.9770
  Mean 332.9275 194.1785    10.43% 0.9644
   Std  85.2276  54.9581     2.86% 0.0103

Cross Validation Summary Table (XGBoost)

  Fold     RMSE      MAE MAAPE (%)     R2
Fold 1 435.1784 256.7362    16.22% 0.9508
Fold 2 389.7735 266.6852    11.46% 0.9666
Fold 3 259.7230 178.3306    16.40% 0.9659
Fold 4 237.0904 14

C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packag


Cross Validation Summary Table (LightGBM)

  Fold     RMSE      MAE MAAPE (%)     R2
Fold 1 416.3711 245.3827    17.76% 0.9550
Fold 2 388.0125 252.9249    10.60% 0.9669
Fold 3 247.0804 167.6631    15.36% 0.9691
Fold 4 230.5649 138.0423    13.42% 0.9758
Fold 5 259.1938 153.1405     8.85% 0.9810
  Mean 308.2445 191.4307    13.20% 0.9696
   Std  86.9417  53.7906     3.58% 0.0099

Cross Validation Summary Table (SVR)

  Fold     RMSE      MAE MAAPE (%)     R2
Fold 1 309.5690 200.6595    18.74% 0.9751
Fold 2 333.7051 228.0694    11.44% 0.9755
Fold 3 237.2736 168.0005    17.05% 0.9715
Fold 4 232.8463 148.7120    16.51% 0.9753
Fold 5 296.5243 194.9762    13.78% 0.9752
  Mean 281.9837 188.0835    15.50% 0.9745
   Std  44.8914  30.6512     2.89% 0.0017
